# 5.6 — Broadcast Variables and Accumulators

**Chapter 5, section 5.11**, and the starting point for **Exercise 9**.

**The question this notebook answers:** the two shared variables are the only sanctioned ways to
move a value across the boundary between driver and executors, and the chapter presents them as a
symmetric pair — one read-only value out, one write-only total back. The symmetry is real, and so
is an asymmetry the chapter is careful about: **a broadcast variable is an optimization that can
be relied upon, and an accumulator is a diagnostic that cannot.**

Both halves are measured here. For the broadcast, what is measured is how many bytes travel and
how often. For the accumulator, what is measured is the overcount — a tally that is exactly right
on a small dataset and quietly wrong on a large one, which is Exercise 9's pipeline that is
"8 % too high in production".

The storage region is deliberately small (`spark.memory.fraction` 0.1, lower even than notebook
5.4 uses), because the accumulator's most important failure needs a cache that does not fit and a
laptop has to be persuaded into that condition.

Runs on a laptop in about a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, json, time, random, tempfile, logging, urllib.request
from urllib.parse import urlparse
import pandas as pd
from pyspark import StorageLevel
from pyspark.serializers import CloudPickleSerializer
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-5.6")
         .master("local[4]")
         .config("spark.driver.memory", "2048m")
         .config("spark.memory.fraction", 0.1)
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
for _lg in ("SQLQueryContextLogger", "DataFrameQueryContextLogger"):
    logging.getLogger(_lg).setLevel(logging.CRITICAL)

pd.set_option("display.width", 210)

_port = urlparse(sc.uiWebUrl).port
UI = f"http://localhost:{_port}/api/v1"
APP = json.load(urllib.request.urlopen(f"{UI}/applications"))[0]["id"]

def ui(path):
    with urllib.request.urlopen(f"{UI}/applications/{APP}{path}", timeout=60) as r:
        return json.load(r)

M_MIB = ui("/executors")[0]["maxMemory"] / 1048576
print("Spark", spark.version, f"| M = {M_MIB:.0f} MiB | cores = {sc.defaultParallelism}")

Spark 4.2.0 | M = 175 MiB | cores = 4


## 1. A broadcast variable is about how many times, not about whether

§5.11.1 opens with the point that is easy to miss: the value a broadcast variable carries *could
be referred to directly, and the program would produce the same answer*. Nothing about
correctness is at stake. The difference is in how many times the value is transmitted — with
every task, or once per executor.

The first thing to establish is therefore what a closure actually weighs. PySpark serializes the
function together with every variable it refers to, using cloudpickle, and that serialized object
is what travels with each task.

In [2]:
ser = CloudPickleSerializer()

def lookup_table(n):
    """An airport-code lookup table of n entries, as an ordinary Python dict."""
    return {f"A{i:05d}": f"City number {i}, with a name long enough to weigh something"
            for i in range(n)}

rows = []
for n in (3, 200, 2_000, 20_000, 200_000):
    codes = lookup_table(n)
    direct = lambda r, codes=codes: codes.get(r, "?")      # captures the whole dict
    bc = sc.broadcast(codes)
    via_bc = lambda r, bc=bc: bc.value.get(r, "?")          # captures a handle
    rows.append({"entries in the table": n,
                 "table alone (KB)": round(len(ser.dumps(codes)) / 1024, 1),
                 "closure, direct (KB)": round(len(ser.dumps(direct)) / 1024, 1),
                 "closure, broadcast (KB)": round(len(ser.dumps(via_bc)) / 1024, 2)})
    # Deliberately not destroyed here; see the last cell of this section for why.

print(pd.DataFrame(rows).to_string(index=False))
print("\nThe third column is what ships with EVERY task. The fourth is what ships with every task")
print("when the table is broadcast: a handle, whose size does not grow with the table.")
print("\nThe chapter quotes the documentation's threshold: tasks above about 20 KiB are worth")
print("examining. A table of only 200 entries already crosses it.")

 entries in the table  table alone (KB)  closure, direct (KB)  closure, broadcast (KB)
                    3               0.2                   0.8                     0.64
                  200              13.8                  14.3                     0.64
                 2000             139.6                 140.1                     0.64
                20000            1415.2                1415.7                     0.64
               200000           14444.7               14445.2                     0.64

The third column is what ships with EVERY task. The fourth is what ships with every task
when the table is broadcast: a handle, whose size does not grow with the table.

The chapter quotes the documentation's threshold: tasks above about 20 KiB are worth
examining. A table of only 200 entries already crosses it.


### What that costs a job, and the Spark UI reading that reveals it

The chapter says large **task deserialization time** in the summary metrics is the same signal
read from the executor's end. Two identical jobs are run below over the same data with the same
number of tasks; they differ only in whether the lookup table travels in the closure or in a
broadcast variable.

In [3]:
KEYS = sc.parallelize([f"A{i % 20000:05d}" for i in range(400_000)], 200).cache()
KEYS.count()

def reduce_stage_metrics(label, fn):
    before = {s["stageId"] for s in ui("/stages")}
    sc.setJobDescription(label)
    t0 = time.time(); result = fn(); secs = time.time() - t0
    new = [s for s in ui("/stages") if s["stageId"] not in before]
    st = max(new, key=lambda s: s["executorRunTime"])
    d = ui(f"/stages/{st['stageId']}/0/taskSummary?quantiles=0.5,1.0")
    return {"job": label, "tasks": st["numTasks"], "seconds": round(secs, 2),
            "median task deser (ms)": round(d["executorDeserializeTime"][0]),
            "max task deser (ms)": round(d["executorDeserializeTime"][1]),
            "total task deser (ms)": st["executorDeserializeTime"],
            "result": result}

# Deliberately BELOW PySpark's automatic-broadcast threshold (about 1 MB), so that the
# closure really is shipped with every task and the comparison has something to measure.
big = lookup_table(2_000)
closure_kb = len(ser.dumps(lambda r, big=big: big.get(r, "?"))) / 1024

direct_row = reduce_stage_metrics(
    "lookup table in the closure",
    lambda: KEYS.map(lambda r, big=big: big.get(r, "?")).filter(lambda s: s != "?").count())

bc = sc.broadcast(big)
bc_row = reduce_stage_metrics(
    "lookup table broadcast",
    lambda: KEYS.map(lambda r, bc=bc: bc.value.get(r, "?")).filter(lambda s: s != "?").count())

print(f"the table serializes to {closure_kb:,.0f} KB; both jobs run"
      f" {direct_row['tasks']} tasks over the same data\n")
print(pd.DataFrame([direct_row, bc_row]).to_string(index=False))
print(f"\nbytes shipped when the table rides in the closure:"
      f" {closure_kb:,.0f} KB x {direct_row['tasks']} tasks ="
      f" {closure_kb * direct_row['tasks'] / 1024:,.1f} MB")
print(f"bytes shipped when it is broadcast               :"
      f" {closure_kb:,.0f} KB x 1 executor = {closure_kb / 1024:,.2f} MB")
print(f"ratio                                            :"
      f" {direct_row['tasks']}x, which is the task count, exactly as the chapter says")

the table serializes to 140 KB; both jobs run 200 tasks over the same data

                        job  tasks  seconds  median task deser (ms)  max task deser (ms)  total task deser (ms)  result
lookup table in the closure    200     0.26                       0                    4                     15   40000
     lookup table broadcast    200     0.26                       0                    2                      4   40000

bytes shipped when the table rides in the closure: 140 KB x 200 tasks = 27.4 MB
bytes shipped when it is broadcast               : 140 KB x 1 executor = 0.14 MB
ratio                                            : 200x, which is the task count, exactly as the chapter says


### Why the table above uses two thousand entries and not twenty thousand

The first attempt at this measurement used the twenty-thousand-entry table and found no difference
at all, for a reason the chapter does not mention: when a closure exceeds about a megabyte,
**PySpark broadcasts it automatically** rather than shipping it with every task. The large table
was already being handled the way a broadcast variable would handle it, so there was nothing left
for an explicit broadcast to improve.

Two things remain true, and they are the reasons to use the primitive deliberately rather than
rely on the automatic path:

* **Below the threshold there is no help at all**, and a table of a few hundred kilobytes over
  thousands of tasks is exactly the case that hurts and is exactly the case that is not
  auto-broadcast.
* **The automatic broadcast lives only as long as the one operation that triggered it.** A
  variable broadcast explicitly is cached on the executor **for the life of the application**, so
  the second, third and tenth job that need it pay nothing. That is the property the chapter is
  pointing at with "cached there for the life of the application", and it is invisible in a
  measurement of a single job.

The cell below shows the size at which the automatic path engages.

In [4]:
print("closure size against PySpark's automatic-broadcast threshold (about 1 MB):\n")
for n in (200, 2_000, 20_000, 200_000):
    t = lookup_table(n)
    kb = len(ser.dumps(lambda r, t=t: t.get(r, "?"))) / 1024
    auto = "auto-broadcast by PySpark" if kb > 1024 else "shipped with every task"
    print(f"   {n:>7,} entries -> closure {kb:>9,.1f} KB   {auto}")
print("\nA table of a few hundred kilobytes is the dangerous size: too small to be rescued")
print("automatically, large enough that thousands of copies of it matter.")

closure size against PySpark's automatic-broadcast threshold (about 1 MB):

       200 entries -> closure      14.3 KB   shipped with every task
     2,000 entries -> closure     140.1 KB   shipped with every task
    20,000 entries -> closure   1,415.7 KB   auto-broadcast by PySpark
   200,000 entries -> closure  14,445.2 KB   auto-broadcast by PySpark

A table of a few hundred kilobytes is the dangerous size: too small to be rescued
automatically, large enough that thousands of copies of it matter.


### The three things that can go wrong with a broadcast variable

§5.11.1 lists four properties. Three of them are failure modes, and all three are shown below:
the value must be an ordinary local object, modifying the original after broadcasting is an error
that produces no message, and `destroy` is permanent where `unpersist` is provisional.

In [5]:
import contextlib, io as _io

# 1. A distributed dataset cannot be broadcast: it is not in the driver's memory to begin with.
#    PySpark prints the pickling traceback itself, so it is captured rather than shown.
try:
    with contextlib.redirect_stderr(_io.StringIO()):
        sc.broadcast(KEYS)
    print("1. broadcasting an RDD: accepted -- unexpected")
except Exception as e:
    print("1. broadcasting an RDD:", type(e).__name__, "-",
          str(e).replace("\n", " ").split("[")[1].split("]")[0])

# 2. Modifying the original after broadcasting: no message, and the executors do not see it.
table = {"BOS": "Boston", "JFK": "New York", "SFO": "San Francisco"}
air = sc.broadcast(table)
table["BOS"] = "SOMEWHERE ELSE"            # no error, no warning
seen = sc.parallelize(["BOS"], 1).map(lambda k, b=air: b.value[k]).collect()[0]
print(f"2. driver's dict now says {table['BOS']!r}; the executors still see {seen!r}")

# 3. unpersist is provisional: a later use simply re-broadcasts.
air.unpersist()
after_unpersist = sc.parallelize(["JFK"], 1).map(lambda k, b=air: b.value[k]).collect()[0]
print(f"3. after unpersist(), a later use re-broadcasts and works: {after_unpersist!r}")

1. broadcasting an RDD: PicklingError - RDD_TRANSFORM_ONLY_VALID_ON_DRIVER
2. driver's dict now says 'SOMEWHERE ELSE'; the executors still see 'Boston'
3. after unpersist(), a later use re-broadcasts and works: 'New York'


### `destroy()` is more permanent than the chapter says

The chapter's fourth property is that `destroy` releases the copies permanently and the variable
cannot be used afterwards. That is true, and on PySpark it is an understatement, which is why this
demonstration is kept to the end of the section.

When a closure capturing a broadcast variable is serialized, PySpark records that broadcast in a
per-thread registry, and **`destroy()` does not remove it**. On the ordinary path this is harmless,
because Spark empties that registry after each RDD it builds. It stops being harmless as soon as a
closure is pickled outside that path — which is exactly what the size measurements at the top of
this notebook did. The dead reference then travels with the *next* job submitted from the thread,
whatever that job is, and fails to serialize.

Both are shown below: first the documented consequence, then the one that catches people out.

In [6]:
# The three failures below are deliberate, and each one makes the JVM log a full stack trace
# through org.apache.spark.util.Utils. That logger is pinned to ERROR in the course's
# log4j2.properties, which is why sc.setLogLevel() cannot quiet it -- an explicit logger level
# overrides the root level. Silence that one logger for the duration of the cell instead.
_Configurator = sc._jvm.org.apache.logging.log4j.core.config.Configurator
_Level = sc._jvm.org.apache.logging.log4j.Level
_Configurator.setLevel("org.apache.spark.util.Utils", _Level.OFF)

air.destroy()

# (a) the documented consequence: the variable itself is gone.
try:
    sc.parallelize(["SFO"], 1).map(lambda k, b=air: b.value[k]).collect()
    print("(a) using it after destroy(): still usable -- unexpected")
except Exception as e:
    print("(a) using it after destroy():", type(e).__name__, "-- as documented")

# (b) an unrelated job, submitted right afterwards, is fine: Spark emptied the registry
#     when it built the RDD above.
try:
    sc.parallelize([1, 2, 3], 2).map(lambda x: x + 1).collect()
    print("(b) an unrelated job afterwards  : fine, on the ordinary path")
except Exception:
    print("(b) an unrelated job afterwards  : fails")

# (c) now the case that bites: pickle a closure OUTSIDE Spark's own path, then destroy.
probe = sc.broadcast({"k": 1})
ser.dumps(lambda r, b=probe: b.value)     # registers it; nothing empties the registry
probe.destroy()
try:
    sc.parallelize([1, 2, 3], 2).map(lambda x: x + 1).collect()
    print("(c) after pickling then destroying: fine -- unexpected")
except Exception as e:
    marker = ("INTERNAL_ERROR_BROADCAST" if "INTERNAL_ERROR_BROADCAST" in str(e)
              else type(e).__name__)
    print(f"(c) after pickling then destroying: the SAME unrelated job now FAILS, {marker}")
    print("    and the message names the destroyed broadcast, not the job that was submitted.")

# The recovery uses a private registry, because there is no public one. In a real program the
# remedy is not to destroy a broadcast variable while the session may still submit work.
sc._pickled_broadcast_vars.clear()
print("\nafter clearing the registry, the unrelated job runs again:",
      sc.parallelize([1, 2, 3], 2).map(lambda x: x + 1).collect())
_Configurator.setLevel("org.apache.spark.util.Utils", _Level.ERROR)

(a) using it after destroy(): Py4JJavaError -- as documented


(b) an unrelated job afterwards  : fine, on the ordinary path
(c) after pickling then destroying: the SAME unrelated job now FAILS, INTERNAL_ERROR_BROADCAST
    and the message names the destroyed broadcast, not the job that was submitted.

after clearing the registry, the unrelated job runs again: [2, 3, 4]


The practical rule: use `unpersist()` to free the copies, and reserve `destroy()` for the end of
the application. The chapter's sentence is right about the variable; what it does not say is that
the damage is not confined to the variable.

## 2. An accumulator, and the three properties that decide what it is for

The return direction is closed: a variable defined in the driver is shipped by copy, and an
executor's modification of its copy never comes back. §5.10.3 showed the counter that stays at
zero. The accumulator is the sanctioned instrument for the one case that needs the boundary
crossed — a running total maintained while a separate transformation does the real work.

The chapter's own example is reproduced first, verbatim: a tally of records that fail to parse.

In [7]:
BAD_RATE = 0.05
lines = ([f"{i},{i*2},{i*3}" for i in range(200_000)]
         + [f"{i},not-a-number,{i}" for i in range(int(200_000 * BAD_RATE))])
CSV = os.path.join(SCRATCH, "ch05-accumulator-input.csv")
with open(CSV, "w") as f:
    f.write("\n".join(lines) + "\n")

true_bad = sum(1 for ln in lines if "not-a-number" in ln)
print(f"{len(lines):,} lines written, of which {true_bad:,} are malformed"
      f" ({true_bad/len(lines):.1%})")

errors = sc.accumulator(0)                 # created on the driver, starts at 0

def parse(line):
    global errors
    try:
        return [float(x) for x in line.split(",")]
    except ValueError:
        errors += 1                        # executors may only add to it
        return []

parsed = sc.textFile(CSV, 8).flatMap(parse)
print(f"\nerrors.value BEFORE any action : {errors.value}   <- property 1: it means nothing yet")
n = parsed.count()                         # an action; only now do the tasks run
print(f"errors.value AFTER  the action : {errors.value:,}   (true answer {true_bad:,})")
print(f"parsed values                  : {n:,}")

210,000 lines written, of which 10,000 are malformed (4.8%)



errors.value BEFORE any action : 0   <- property 1: it means nothing yet


errors.value AFTER  the action : 10,000   (true answer 10,000)
parsed values                  : 600,000


### Property 2: exactly-once holds inside an action, not inside a transformation

The update above runs inside `flatMap`, which is a transformation. Spark guarantees each task's
update is applied exactly once only when the update occurs inside an **action**; inside a
transformation, a task that runs again applies its update again.

Tasks run again for ordinary reasons, and the cheapest one to arrange is the most common one in
practice: **a second action over an uncached dataset**.

In [8]:
acc = sc.accumulator(0)

def parse2(line):
    global acc
    try:
        return [float(x) for x in line.split(",")]
    except ValueError:
        acc += 1
        return []

uncached = sc.textFile(CSV, 8).flatMap(parse2)

rows = []
for k in (1, 2, 3):
    uncached.count()
    rows.append({"actions taken": k, "accumulator reads": acc.value,
                 "true answer": true_bad,
                 "overcount": f"{acc.value / true_bad - 1:+.0%}"})
print(pd.DataFrame(rows).to_string(index=False))
print("\nNothing failed. No executor was lost. The dataset was simply traversed three times,")
print("and the accumulator counted the same malformed records three times.")

 actions taken  accumulator reads  true answer overcount
             1              10000        10000       +0%
             2              20000        10000     +100%
             3              30000        10000     +200%

Nothing failed. No executor was lost. The dataset was simply traversed three times,
and the accumulator counted the same malformed records three times.


### Property 3: caching does not repair it

The reasonable-sounding remedy is to cache the parsed dataset, so that the second action does not
recompute it. It works — until the dataset is larger than the memory available to hold it, at
which point the partitions that did not fit are recomputed on every access, **along with their
accumulator updates**.

This is the chapter's sharpest warning and the whole of Exercise 9: *a job whose accumulator is
correct on a small dataset that fits comfortably in the cache and quietly wrong on a large one
that does not.* Both cases are run below on the same code.

In [9]:
def run_cached(label, n_rows, n_parts):
    """Parse a file of n_rows with an accumulator inside the transformation, cache it,
    and take two actions. The second action should be free -- if the cache held."""
    path = os.path.join(SCRATCH, f"ch05-acc-{label}.csv")
    # The padding is random hex rather than a repeated character, and that is not decoration.
    # PySpark caches RDDs with spark.rdd.compress enabled, and a column repeating one character
    # compresses by a factor of thirty -- which makes any dataset fit and hides the effect this
    # cell exists to show. Incompressible padding is what forces the cache to overflow.
    rnd = random.Random(0)
    def pad():
        return f"{rnd.getrandbits(1024):0256x}"
    # The malformed rows are spread evenly through the file rather than appended at the end.
    # Concentrating them in the last partitions makes the result depend on which partitions
    # happen to be evicted, which is not the effect being demonstrated.
    every = int(1 / BAD_RATE)
    bad = 0
    with open(path, "w") as f:
        for i in range(n_rows):
            if i % every == 0:
                f.write(f"{i},not-a-number,{pad()}\n")
                bad += 1
            else:
                f.write(f"{i},{i*2},{pad()}\n")

    a = sc.accumulator(0)

    def p(line, a=a):
        # a.add(1) rather than "global a; a += 1": the accumulator here is a local of
        # run_cached, and += would rebind the name instead of adding to the accumulator.
        # The two forms are equivalent; only the scoping differs.
        parts = line.split(",")
        try:
            float(parts[1])
            return [(parts[0], parts[2])]
        except ValueError:
            a.add(1)
            return []

    rdd = sc.textFile(path, n_parts).flatMap(p)
    rdd.persist(StorageLevel.MEMORY_ONLY)
    rdd.count()                                   # first action: materializes the cache
    first = a.value
    stored = [r for r in ui("/storage/rdd") if r["numPartitions"] == n_parts]
    frac = (stored[-1]["numCachedPartitions"] / stored[-1]["numPartitions"]) if stored else float("nan")
    cached_mb = round(stored[-1]["memoryUsed"] / 1e6, 1) if stored else float("nan")
    rdd.count()                                   # second action: should cost nothing
    second = a.value
    rdd.unpersist(blocking=True)
    return {"dataset": label, "true bad rows": bad, "cached MB": cached_mb,
            "fraction cached": f"{frac:.0%}",
            "after 1 action": first, "after 2 actions": second,
            "overcount": f"{second / bad - 1:+.0%}"}

small_case = run_cached("small", 50_000, 8)
big_case = run_cached("large", 1_200_000, 40)
print(pd.DataFrame([small_case, big_case]).to_string(index=False))
print(f"\nM on this session is {M_MIB:.0f} MiB.")
print("The code is identical in both rows. The only difference is whether the dataset fitted.")
print("The overcount is the uncached fraction: the partitions that were not held were")
print("recomputed for the second action, and recomputing them ran their accumulator updates")
print("again. Exercise 9's pipeline is this with a smaller overflow -- 8% of the data missing")
print("from the cache gives a tally 8% too high.")
print("This is the class of error the chapter calls the most difficult to detect, because the")
print("small case is the one that gets tested.")

dataset  true bad rows  cached MB fraction cached  after 1 action  after 2 actions overcount
  small           2500       11.8            100%            2500             2500       +0%
  large          60000      177.9             62%           60000            82433      +37%

M on this session is 175 MiB.
The code is identical in both rows. The only difference is whether the dataset fitted.
The overcount is the uncached fraction: the partitions that were not held were
recomputed for the second action, and recomputing them ran their accumulator updates
again. Exercise 9's pipeline is this with a smaller overflow -- 8% of the data missing
from the cache gives a tally 8% too high.
This is the class of error the chapter calls the most difficult to detect, because the
small case is the one that gets tested.


## 3. What to do instead

The conclusion §5.11.2 draws is not that accumulators should be avoided but that their purpose
should be stated correctly: they are instruments for **monitoring and diagnosis**, not for
producing results. Three forms are compared below.

In [10]:
# (a) the accumulator inside a transformation -- a diagnostic, not a result
a_transform = sc.accumulator(0)
def p_transform(line):
    global a_transform
    try:
        return [float(x) for x in line.split(",")]
    except ValueError:
        a_transform += 1
        return []
d = sc.textFile(CSV, 8).flatMap(p_transform)
d.count(); d.count()

# (b) the accumulator inside an ACTION -- exactly-once, per the documentation
a_action = sc.accumulator(0)
def p_action(line):
    global a_action
    try:
        [float(x) for x in line.split(",")]
    except ValueError:
        a_action += 1
raw = sc.textFile(CSV, 8)
raw.foreach(p_action)
raw.foreach(p_action)          # two actions, so this one legitimately doubles

# (c) an aggregation whose result is returned by an action -- exact by construction
def is_bad(line):
    try:
        [float(x) for x in line.split(",")]
        return False
    except ValueError:
        return True
exact = sc.textFile(CSV, 8).filter(is_bad).count()

print(pd.DataFrame([
    {"form": "(a) accumulator in a transformation, 2 actions",
     "value": a_transform.value, "true": true_bad, "exact?": a_transform.value == true_bad},
    {"form": "(b) accumulator in foreach, called twice",
     "value": a_action.value, "true": true_bad, "exact?": a_action.value == true_bad},
    {"form": "(c) filter().count(), one action",
     "value": exact, "true": true_bad, "exact?": exact == true_bad},
]).to_string(index=False))
print("\n(b) is not a counter-example to exactly-once: two actions ran, and each applied its")
print("updates once. Exactly-once is a guarantee per action, not per program.")
print("(c) is what Exercise 9(b) asks for. What it costs is a second pass over the data --")
print("which is the price of an exact figure, and is why the accumulator is tempting.")

                                          form  value  true  exact?
(a) accumulator in a transformation, 2 actions  20000 10000   False
      (b) accumulator in foreach, called twice  20000 10000   False
              (c) filter().count(), one action  10000 10000    True

(b) is not a counter-example to exactly-once: two actions ran, and each applied its
updates once. Exactly-once is a guarantee per action, not per program.
(c) is what Exercise 9(b) asks for. What it costs is a second pass over the data --
which is the price of an exact figure, and is why the accumulator is tempting.


### The appropriate use, made automatic

Exercise 9(c) asks for the use the original accumulator remains entirely right for, and for how to
turn it into a check. The chapter gives the production practice: compare the tally against a
threshold and fail the job when the proportion of bad records is too high. The accumulator's
inexactness does not matter here, because the comparison is against a threshold rather than a
specification.

In [11]:
def ingest_with_quality_gate(path, max_bad_fraction=0.10):
    """Parse, tally malformed records as a diagnostic, and refuse the batch if too many."""
    bad = sc.accumulator(0)
    total = sc.accumulator(0)

    def parse_row(line, bad=bad, total=total):
        total.add(1)
        try:
            return [tuple(float(x) for x in line.split(","))]
        except ValueError:
            bad.add(1)
            return []

    out = sc.textFile(path, 8).flatMap(parse_row)
    kept = out.count()                   # the single action that runs the tally
    seen, rejected = total.value, bad.value
    fraction = rejected / seen if seen else 0.0
    verdict = "ACCEPTED" if fraction <= max_bad_fraction else "REJECTED"
    return {"rows seen": seen, "rows kept": kept, "malformed (approx)": rejected,
            "malformed fraction": f"{fraction:.2%}",
            "threshold": f"{max_bad_fraction:.0%}", "verdict": verdict}

print(pd.DataFrame([ingest_with_quality_gate(CSV, 0.10),
                    ingest_with_quality_gate(CSV, 0.01)]).to_string(index=False))
print("\nThe tally is approximate and the decision is not, because a figure a few percent high")
print("does not move a judgment against a threshold of ten percent. That is the difference")
print("between a diagnostic and a result.")

 rows seen  rows kept  malformed (approx) malformed fraction threshold  verdict
    210000     200000               10000              4.76%       10% ACCEPTED
    210000     200000               10000              4.76%        1% REJECTED

The tally is approximate and the decision is not, because a figure a few percent high
does not move a judgment against a threshold of ten percent. That is the difference
between a diagnostic and a result.


### One capability a PySpark user should not go looking for

§5.11.2 closes with two further features, and records a limitation of the second. Accumulators may
be given names, in which case Spark displays them in the Stages tab beside the tasks that modified
them — **except in Python**, where the named-accumulator view is not supported. The PySpark API
reflects this: `sc.accumulator` takes a value and an optional `AccumulatorParam`, and no name.

In [12]:
import inspect
print("signature:", "sc.accumulator" + str(inspect.signature(sc.accumulator)))
print("\nThere is no name parameter, so there is nothing to search the Stages tab for.")
print("A named diagnostic in PySpark is a print statement in the driver after the action,")
print("or a metric written somewhere the job's operator will look.")

# Custom accumulator types are supported, and are the other half of that paragraph.
from pyspark.accumulators import AccumulatorParam

class VectorAccumulatorParam(AccumulatorParam):
    """An accumulator over a fixed-length list: zero value plus an associative addition."""
    def zero(self, value):
        return [0] * len(value)
    def addInPlace(self, a, b):
        return [x + y for x, y in zip(a, b)]

counts = sc.accumulator([0, 0, 0], VectorAccumulatorParam())

def classify(line):
    global counts
    parts = line.split(",")
    try:
        v = float(parts[1])
        counts += [1, 0, 0] if v >= 0 else [0, 1, 0]
    except ValueError:
        counts += [0, 0, 1]

sc.textFile(CSV, 8).foreach(classify)
print(f"\ncustom accumulator over three categories: {counts.value}")
print("   [non-negative, negative, malformed]  -- one pass, three tallies, inside an action")

signature: sc.accumulator(value: ~T, accum_param: Optional[ForwardRef('AccumulatorParam[T]')] = None) -> 'Accumulator[T]'

There is no name parameter, so there is nothing to search the Stages tab for.
A named diagnostic in PySpark is a print statement in the driver after the action,
or a metric written somewhere the job's operator will look.

custom accumulator over three categories: [200000, 0, 10000]
   [non-negative, negative, malformed]  -- one pass, three tallies, inside an action


## Conclusion

The two primitives are symmetric in what they do and not in how far they can be trusted, and that
asymmetry is the thing to carry away.

What this notebook establishes by running it:

1. **A closure carries everything the function refers to.** A twenty-thousand-entry lookup table
   made the closure megabytes; broadcasting it left a handle whose size does not grow with the
   table. The chapter's 20 KiB threshold is crossed by a table of a few hundred entries.
2. **PySpark auto-broadcasts closures above about a megabyte**, so the single-job measurement
   understates nothing and overstates nothing — the reasons to broadcast deliberately are the
   sub-threshold case and the fact that an explicit broadcast is cached for the life of the
   application rather than for one operation.
3. **Modifying a broadcast value after broadcasting produces no message** and no effect on the
   executors; `destroy` is permanent and `unpersist` is not.
4. **An accumulator read before an action is zero**, and an accumulator updated inside a
   transformation counts once per *execution*, not once per record: three actions over an uncached
   dataset tripled it.
5. **Caching does not repair that.** The same code was exact on a dataset that fitted in memory
   and over by the uncached fraction on one that did not — the small case being, as the chapter
   says, the one that gets tested.
6. **An exact figure costs a pass over the data.** `filter().count()` is exact by construction,
   and that second traversal is the price the accumulator was trying to avoid.

*Chapter section:* §5.11 (shared variables). *Exercise 9* is answered in section 2: (a) by the two
mechanisms in the repeated-action and partial-cache tables, (b) by form (c) of section 3, and (c)
by the quality gate.